In [1]:
import os
from odps import ODPS
import numpy as np
import pandas as pd
from datetime import datetime,timedelta
import os

##initialize odps
o = ODPS(
    # （推荐）确保已设置环境变量。
    # 确保ALIBABA_CLOUD_ACCESS_KEY_ID环境变量设置为用户 Access Key ID。
    access_id=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_ID'),
    
    # 确保ALIBABA_CLOUD_ACCESS_KEY_SECRET环境变量设置为用户Access Key Secret。
    secret_access_key=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_SECRET'),
    project='xyf_jingying_dev',
    endpoint='https://service.cn-beijing.maxcompute.aliyun.com/api',
)

In [2]:
# 修改训练集开始、结束日期设置为昨天
date_train_start = '2024-01-01'
date_train_end = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

# 修改预测月截止日期
date_pred_start = datetime.today().strftime('%Y-%m-%d')
date_pred_end = '2026-07-31'

# 生成文件路径
today = datetime.today()
next_month = today.month % 12 + 1
mmdd = today.strftime("%m%d")
base_dir = f"D:/3.月度放款预测&资产日度调整/26年{next_month}月放款预测/{mmdd}/"
os.makedirs(base_dir, exist_ok=True)  # exist_ok=False（默认）：目录已存在会抛 FileExistsError; exist_ok=True：目录已存在不报错；需要的上级目录会自动创建
train_data_path = os.path.join(base_dir, "train_data.xlsx")
test_data_path = os.path.join(base_dir, "test_data.xlsx")
pred_y_path  = os.path.join(base_dir, "pred_y.xlsx")

In [3]:
query1 = '''
SELECT  a.crt_dt
       ,a.`老客资产量`                                                 AS amt
       ,DATEDIFF(substr(a.crt_dt,1,10),'2023-01-01')+1                AS line_num
       ,CASE WHEN b.dt_week = 1 THEN 1  ELSE 0 END                    AS if_mon
       ,CASE WHEN b.dt_week = 2 THEN 1  ELSE 0 END                    AS if_tues
       ,CASE WHEN b.dt_week = 3 THEN 1  ELSE 0 END                    AS if_wed
       ,CASE WHEN b.dt_week = 4 THEN 1  ELSE 0 END                    AS if_thur
       ,CASE WHEN b.dt_week = 5 THEN 1  ELSE 0 END                    AS if_fri
       ,CASE WHEN b.dt_week = 6 THEN 1  ELSE 0 END                    AS if_sat
       ,CASE WHEN b.dt_week = 7 THEN 1  ELSE 0 END                    AS if_sun
       ,CASE WHEN (b.dt_week = 6 OR b.dt_week = 7) THEN 1  ELSE 0 END AS if_wked
       ,CASE WHEN substr(a.crt_dt,9,2) = '28' THEN 1  ELSE 0 END      AS if_28th
       ,CASE WHEN substr(a.crt_dt,9,2) = '10' THEN 1  ELSE 0 END      AS if_10th
       ,CASE WHEN substr(a.crt_dt,9,2) = '20' THEN 1  ELSE 0 END      AS if_20th
       ,CASE WHEN substr(a.crt_dt,9,2) <= '03' THEN 1  ELSE 0 END     AS if_month_begin
       ,CASE WHEN substr(a.crt_dt,9,2) > '28' THEN 1  ELSE 0 END      AS if_month_end
       ,b.is_holiday --是否节假日
       ,a.`到期人数`                                                      AS due_cnt
       ,a.`到期金额`                                                      AS due_amt
       ,(a.`通过`)/a.`发起人数`                                           AS pass_rate
FROM
(
	SELECT  a.crt_dt
	       ,a.`发起人数`
	       ,a.`通过`
	       ,a.`老客资产量`
	       ,b.`到期人数`
	       ,b.`到期金额`
	FROM
	( -- 创单人数 & 通过率 & 资产量
		SELECT  DATE(first_order_time) crt_dt
		       ,COUNT(DISTINCT CASE WHEN app = inner_app THEN cust_no END)                          AS `发起人数` --仅在APP渠道上的发起
		       ,COUNT(DISTINCT CASE WHEN app = inner_app AND risk_status = 'pass' THEN cust_no END) AS `通过` --仅在APP渠道上的发起
		       ,SUM(CASE WHEN risk_status = 'pass' THEN order_amt ELSE 0 END)                       AS `老客资产量` --包含api复贷的资产量
		FROM
		(
			SELECT  a.cust_no
			       ,a.order_amt --大额拆单上线后，用order_amt统计资产量
			       ,a.loan_amt
			       ,a.first_order_time
			       ,a.risk_status
			       ,a.loan_status
			       ,a.loan_flag
			       ,a.app
			       ,a.inner_app
			FROM xyf_dws.dws_inloan_user_order_df a
			WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
			AND DATE(a.first_order_time) >= '{date_train_start}'
            AND DATE(a.first_order_time) <= '{date_train_end}'
			AND a.app IN ('xyf01', 'fxk') 
			-- AND a.app = a.inner_app --限制app渠道上的复贷人数，case when中限制
			AND a.loan_flag IN ('加贷', '复贷') 
		)
		GROUP BY  DATE(first_order_time)
	) a
	LEFT JOIN
	( -- 账单日到期人数 
		SELECT  DATE(date_due)                                AS dt
		       ,COUNT(DISTINCT cust_no)                       AS `到期人数`
		       ,SUM(should_pay_principal+should_pay_interest) AS `到期金额`
		FROM xyf_dwd.dwd_repay_loan_repay_plan_df
		WHERE pt = max_pt('xyf_dwd.dwd_repay_loan_repay_plan_df')
		AND DATE(date_due) >= '{date_train_start}'
		AND DATE(date_due) <= '{date_train_end}'
		AND app IN ('xyf01', 'fxk') --不用限制inner_app 
		GROUP BY  DATE(date_due)
	) b
	ON a.crt_dt = b.dt
) a
LEFT JOIN xyf_dim.dim_pub_date b
ON a.crt_dt = b.day_id_iso
'''.format(date_train_start=date_train_start, date_train_end=date_train_end)

query2 = '''
SELECT  day_id_iso                                                  AS crt_dt
       ,DATEDIFF(substr(day_id_iso,1,10),'2023-01-01')+1            AS line_num
       ,CASE WHEN dt_week = 1 THEN 1  ELSE 0 END                    AS if_mon
       ,CASE WHEN dt_week = 2 THEN 1  ELSE 0 END                    AS if_tues
       ,CASE WHEN dt_week = 3 THEN 1  ELSE 0 END                    AS if_wed
       ,CASE WHEN dt_week = 4 THEN 1  ELSE 0 END                    AS if_thur
       ,CASE WHEN dt_week = 5 THEN 1  ELSE 0 END                    AS if_fri
       ,CASE WHEN dt_week = 6 THEN 1  ELSE 0 END                    AS if_sat
       ,CASE WHEN dt_week = 7 THEN 1  ELSE 0 END                    AS if_sun
       ,CASE WHEN (dt_week = 6 or dt_week = 7) THEN 1  ELSE 0 END   AS if_wked
       ,CASE WHEN substr(day_id_iso,9,2) = '28' THEN 1  ELSE 0 END  AS if_28th
       ,CASE WHEN substr(day_id_iso,9,2) = '10' THEN 1  ELSE 0 END  AS if_10th
       ,CASE WHEN substr(day_id_iso,9,2) = '20' THEN 1  ELSE 0 END  AS if_20th
       ,CASE WHEN substr(day_id_iso,9,2) <= '03' THEN 1  ELSE 0 END AS if_month_begin
       ,CASE WHEN substr(day_id_iso,9,2) > '28' THEN 1  ELSE 0 END  AS if_month_end
       ,is_holiday
       ,b.`到期人数`                                                 AS due_cnt
FROM
(
	SELECT  *
	FROM xyf_dim.dim_pub_date
	WHERE day_id_iso BETWEEN '{date_pred_start}' AND '{date_pred_end}'   --修改月度更新结束日期
) a
JOIN
(
	-- 账单日到期人数 
	SELECT  date(date_due) dt
	       ,COUNT(distinct cust_no) `到期人数`
	       ,SUM(should_pay_principal+should_pay_interest) `到期金额`
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = max_pt('xyf_dwd.dwd_repay_loan_repay_plan_df')
	AND date(date_due) BETWEEN '{date_pred_start}' AND '{date_pred_end}'   --修改月度更新结束日期
	AND app IN ('xyf01', 'fxk') --不用限制inner_app 
	GROUP BY  date(date_due)
) b
ON a.day_id_iso = b.dt
'''.format(date_pred_start = date_pred_start, date_pred_end = date_pred_end)


In [4]:
train_data = o.execute_sql(query1).open_reader().to_pandas()
test_data = o.execute_sql(query2).open_reader().to_pandas()

# 转换所有数值列
numeric_columns = ['amt', 'line_num', 'if_mon', 'if_tues', 'if_wed', 'if_thur', 'if_fri', 
                   'if_sat', 'if_sun', 'if_wked', 'if_28th', 'if_10th', 'if_20th', 
                   'if_month_begin', 'if_month_end', 'is_holiday', 'due_cnt', 'due_amt', 'pass_rate']
for col in numeric_columns:
    if col in train_data.columns:
        train_data[col] = pd.to_numeric(train_data[col], errors='coerce').astype(float)

for col in numeric_columns:
    if col in test_data.columns:
        test_data[col] = pd.to_numeric(test_data[col], errors='coerce').astype(float)

# 处理时间列并按照时间排序
train_data["crt_dt"] = pd.to_datetime(train_data["crt_dt"])
train_data = train_data.sort_values("crt_dt").reset_index(drop=True)

test_data["crt_dt"] = pd.to_datetime(test_data["crt_dt"])
test_data = test_data.sort_values("crt_dt").reset_index(drop=True)

# 把训练集和测试集按照日期拼接起来
all_data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)


In [5]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

# ============ 对系数序列做 ADF 检验并汇总 ============ 
def adf_test(df_or_series, cols: list[str] | None = None, title: str = "ADF 检验结果汇总"):
    """
    合并版 ADF：
    - 传入 Series：返回 (adf_stat, p_value, critical_5)
    - 传入 DataFrame + cols：逐列做 ADF，打印汇总表，并返回汇总 DataFrame
    """
    # 情况1：单序列
    if cols is None:
        series = df_or_series
        result = adfuller(series, autolag="AIC")
        adf_stat = result[0]
        p_value = result[1]
        critical_5 = result[4]["5%"]
        return adf_stat, p_value, critical_5

    # 情况2：多列汇总
    df = df_or_series
    adf_results = {}
    for c in cols:
        result = adfuller(df[c], autolag="AIC")
        adf_results[c] = {
            "ADF Statistic": result[0],
            "p-value": result[1],
            "5% Critical Value": result[4]["5%"],
        }

    adf_df = pd.DataFrame(adf_results).T
    # print(f"\n====== {title} ======")
    # print(adf_df)
    return adf_df
# ADF检验统计量（ADF statistic）
# 位置：result[0]
# 说明：这是ADF检验得到的检验统计量，用于判断序列是否存在单位根。
# p值（p-value）
# 位置：result[1]
# 说明：与ADF统计量对应的p值，用于衡量原假设（存在单位根）的拒绝程度。p值越小，拒绝原假设的证据越强。
# 使用的滞后阶数（usedlag）
# 位置：result[2]
# 说明：ADF回归中实际使用的滞后阶数。这个值可能是通过autolag参数自动选择的最优阶数，或者是你手动指定的最大滞后阶数的一部分。
# 有效观测值数目（nobs）
# 位置：result[3]
# 说明：在进行ADF回归时，考虑滞后项后剩余的有效样本量。
# 临界值字典（critical values）
# 位置：result[4]
# 说明：这是一个字典，包含在常用显著性水平（如1%、5%、10%）下的临界值。例如，result[4]['5%'] 表示在5%显著性水平下的临界值。
# 最优信息准则值（icbest）
# 位置：result[5]
# 说明：当使用了自动滞后选择（如设置 autolag='AIC' 或其他准则）时，此值表示选择滞后阶数时所使用的最佳信息准则值（如最小的AIC值）。
# 其他可选项

def var_model_pipeline_fill(
    all_data: pd.DataFrame,
    feat_cols: list[str],
    split_date: str,
    lags: int = 2,
    fill_cols: list[str] | None = None,
):
    """
    用 VAR 在 all_data 上做“训练(<=split_date之前) -> 外推(>=split_date)”并回填缺失值。

    参数
    - all_data: 必须包含列 crt_dt，以及 feat_cols（测试期可以为 NaN）
    - feat_cols: 用于VAR建模的变量列（必须都是数值）
    - split_date: 分割日期（例如 date_pred_start），训练集取 crt_dt < split_date
    - lags: VAR(p) 的 p
    - fill_cols: 需要回填的列；默认回填 feat_cols 内在预测期为 NaN 的列

    返回
    - filled: 回填后的 all_data（复制）
    - pred_df: 预测期（crt_dt>=split_date）对应的 VAR 外推结果
    - var_model: 训练好的 VAR Results
    """
    df = all_data.copy()
    if "crt_dt" not in df.columns:
        raise ValueError("all_data 必须包含 'crt_dt' 列")

    # 只取 VAR 需要的列，确保为数值
    use_cols = list(dict.fromkeys(feat_cols))  # 去重保序
    for c in use_cols:
        if c not in df.columns:
            raise ValueError(f"all_data 缺少特征列: {c}")

    # 分割
    split_date = pd.to_datetime(split_date)
    train_mask = df["crt_dt"] < split_date
    test_mask = ~train_mask

    train_set = df.loc[train_mask, use_cols].copy()

    if len(train_set) <= lags:
        raise ValueError(f"训练样本不足：len(train_set)={len(train_set)}，lags={lags}")

    # 训练 VAR
    model = VAR(train_set)
    var_model = model.fit(lags)

    n_steps = int(test_mask.sum())
    if n_steps <= 0:
        # 没有预测期，直接返回
        return df, pd.DataFrame(index=df.loc[test_mask, "crt_dt"]), var_model

    initial_input = train_set.values[-lags:]
    fcst = var_model.forecast(initial_input, steps=n_steps)

    pred_index = df.loc[test_mask, "crt_dt"].values
    pred_df = pd.DataFrame(fcst, columns=use_cols, index=pred_index)
    pred_df.index.name = "crt_dt"

    # 回填：只填充预测期内的 NaN
    filled = df.copy()
    if fill_cols is None:
        fill_cols = use_cols

    # 只回填预测期存在的列
    fill_cols = [c for c in fill_cols if c in use_cols]

    for c in fill_cols:
        target_idx = filled.index[test_mask]
        # 对应预测值（按同样顺序）
        pred_vals = pred_df[c].values
        # 仅填 NaN
        mask_nan = filled.loc[target_idx, c].isna().values
        if mask_nan.any():
            filled.loc[target_idx[mask_nan], c] = pred_vals[mask_nan]

    return filled, pred_df, var_model

In [6]:
# 1) VAR 只外推“原始列”
var_base_cols = ["pass_rate", "due_amt"]   # due_cnt未来能取到就别放进来
# adf_test(train_data, cols=var_base_cols)
# model = VAR(call_OLS[coef_names])  

# # 方法一：直接用 select_order(maxlags=12) 来查看各个信息准则（AIC、BIC、FPE、HQIC）
# # （注意：select_order只能直接得到 AIC/BIC/FPE/HQIC 四个指标，LR 需另行计算）
# lag_order_selection = model.select_order(maxlags=12)
# print("\n====== VAR 滞后阶数选择（内置AIC/BIC等） ======")
# print(lag_order_selection.summary())

# # 获取 BIC 对应的最优阶数
# optimal_lag_bic = lag_order_selection.selected_orders['bic']
# print(f"BIC 选择的最优滞后阶数: {optimal_lag_bic}")

all_data_filled, var_pred_df, var_model = var_model_pipeline_fill(
    all_data=all_data,
    feat_cols=var_base_cols,          # 需要被外推的特征列
    split_date=date_pred_start,       # 训练用 split_date 之前
    lags=2,
    fill_cols=var_base_cols,          # 回填这些列（只回填NaN）
)

print("VAR外推完成，预测期行数:", len(var_pred_df))
# 看看预测期被回填后的缺失情况
test_mask = pd.to_datetime(all_data_filled["crt_dt"]) >= pd.to_datetime(date_pred_start)
print("预测期剩余NaN数量(按列):")
print(all_data_filled.loc[test_mask, var_base_cols].isna().sum().sort_values(ascending=False).head(20))

VAR外推完成，预测期行数: 33
预测期剩余NaN数量(按列):
pass_rate    0
due_amt      0
dtype: int64


In [7]:
# ========= 2) 特征衍生 =========
all_data_filled["dow"] = all_data_filled["crt_dt"].dt.dayofweek + 1 # .dt.dayofweek 会把每个日期转换成“星期几”的数字， 1~7, Monday=1

# 滞后项
for lag in [7, 14, 28]:
    all_data_filled[f"amt_lag_{lag}"] = all_data_filled["amt"].shift(lag)
    all_data_filled[f"due_cnt_lag_{lag}"] = all_data_filled["due_cnt"].shift(lag)
    all_data_filled[f"pass_rate_lag_{lag}"] = all_data_filled["pass_rate"].shift(lag)

# 滚动均值
for win in [7, 14, 28]:
    all_data_filled[f"amt_roll_mean_{win}"] = all_data_filled["amt"].shift(1).rolling(win).mean()
    all_data_filled[f"due_cnt_roll_mean_{win}"] = all_data_filled["due_cnt"].shift(1).rolling(win).mean()
    all_data_filled[f"pass_rate_roll_mean_{win}"] = all_data_filled["pass_rate"].shift(1).rolling(win).mean()

# 生成：截至昨日，同周几的历史均值
# all_data_filled["amt_dow_mean"] = (
#     all_data_filled.groupby("dow")["amt"]
#         .apply(lambda s: s.shift(1).expanding().mean())
#         .reset_index(level=0, drop=True)
# )
# all_data_filled["due_cnt_dow_mean"] = (
#     all_data_filled.groupby("dow")["due_cnt"]
#         .apply(lambda s: s.shift(1).expanding().mean())
#         .reset_index(level=0, drop=True)
# )
# all_data_filled["pass_rate_dow_mean"] = (
#     all_data_filled.groupby("dow")["pass_rate"]
#         .apply(lambda s: s.shift(1).expanding().mean())
#         .reset_index(level=0, drop=True)
# )

# 可用的基础日历特征（预测期可直接从日期得到）
#cal_feats = ["if_10th","if_20th","if_month_begin", "if_month_end","is_holiday"]
cal_feats = ['if_mon', 'if_tues', 'if_wed', 'if_thur', 'if_fri', 'if_sat', 'if_sun', 'if_wked', 
            'if_10th', 'if_20th', 'if_28th', 'if_month_begin', 'if_month_end', 'is_holiday']

# 训练用特征列
feat_cols = [
    "amt_lag_7","amt_lag_14","amt_lag_28",
    "amt_roll_mean_7","amt_roll_mean_14","amt_roll_mean_28",
    #"amt_dow_mean",
    "due_cnt_lag_7","due_cnt_lag_14","due_cnt_lag_28",
    "due_cnt_roll_mean_7","due_cnt_roll_mean_14","due_cnt_roll_mean_28",
    #"due_cnt_dow_mean",
    "pass_rate_lag_7","pass_rate_lag_14","pass_rate_lag_28",
    "pass_rate_roll_mean_7","pass_rate_roll_mean_14","pass_rate_roll_mean_28",
    #"pass_rate_dow_mean",
    "due_amt" , "pass_rate"
]
# 预测列
feature_cols = cal_feats + feat_cols

# ======= # 有些列预测期缺省，用最后一个可用值往后填充（预测期） =======
pred_mask = all_data_filled["crt_dt"] >= pd.to_datetime(date_pred_start)
all_data_filled.loc[pred_mask, feat_cols ] = (
    all_data_filled.loc[pred_mask, feat_cols].ffill()
)

train_df = all_data_filled.dropna(subset = feature_cols).copy()

train_df_path = os.path.join(base_dir, "train_df.xlsx")
train_df.to_excel(train_df_path, index=False)

In [8]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
import joblib

def lightgbm_predict(train_set, test_set, feature_cols, parameters, n_estimators=50, random_state=21):
    """
    使用 LightGBM 对指定的参数进行预测。

    参数：
    - train_set: DataFrame，训练数据集，包含目标参数和其滞后特征。#22
    - test_set: DataFrame，测试数据集，包含滞后特征。
    - feature_cols: List，特征列的名称。
    - parameters: List，需要预测的参数列表。
    - n_estimators: int，树的数量，默认值为100。
    - random_state: int，随机种子，确保结果可重复，默认值为42。

    返回：
    - pred_df: DataFrame，包含预测结果，索引与 test_set 一致。
    """
    # 存储训练好的模型
    models = {}

    # 对每个参数分别训练模型，使用所有参数的滞后项作为特征
    for param in parameters:
        X_train = train_set[feature_cols]
        y_train = train_set[param]  # 当前的目标值为某个参数的实际值

        # 建立 LightGBM 模型
        model = LGBMRegressor(n_estimators=n_estimators, random_state=random_state)
        model.fit(X_train, y_train)

        # 将模型保存到字典中
        models[param] = model

    # 使用选用的预测特征进行预测
    X_test = test_set[feature_cols]

    # 初始化预测结果 DataFrame
    pred_df = pd.DataFrame(index=test_set.index)

    # 对每个参数进行预测
    for param in parameters:
        model = models[param]
        pred_values = model.predict(X_test)
        pred_df[param] = pred_values

        # 计算预测误差
        mse = mean_squared_error(test_set[param], pred_df[param])
        print(f'Mean Squared Error for {param}: {mse}')

    return pred_df


data = pd.read_excel(train_df_path)
data.set_index('crt_dt', inplace=True)
split_date = '2025-11-01'

# 数据集划分
train_set = data[data.index < split_date].copy()
test_set = data[data.index >= split_date].copy()
# pred_set = data[data.index >= date_pred_start].copy()

# **超参数搜索空间**
param_grid = {
    'num_leaves': [10, 20, 31, 50],
    'max_depth': [3, 5, 10, None],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'min_child_samples': [5, 10, 20]
}

# **存储最优参数**
best_params = {}
models = {}
parameters = ['amt']

# **对每个回归参数进行超参数优化**
for param in parameters:
    X_train = train_set[feature_cols]
    y_train = train_set[param]
    
    lgbm = LGBMRegressor(random_state=22)
    grid_search = GridSearchCV(lgbm, param_grid, scoring='neg_mean_squared_error', cv=3, n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_params[param] = grid_search.best_params_
    
    # 训练最优模型
    best_lgbm = LGBMRegressor(**grid_search.best_params_, random_state=22)
    best_lgbm.fit(X_train, y_train)
    
    models[param] = best_lgbm

# 在测试集上进行预测
X_test_pred = test_set[feature_cols]
pred_test_df = pd.DataFrame(index=test_set.index)
for param in parameters:
    pred_test_df[param] = models[param].predict(X_test_pred)

# # 在预测集上进行预测
# X_pred = pred_set[feature_cols]
# pred_y_df = pd.DataFrame(index=pred_set.index)
# for param in parameters:
#     pred_y_df[param] = models[param].predict(X_pred)


# **保存超参数和 RMSE 结果**
result_df = pd.DataFrame({
    "参数": parameters,
    # "测试集 RMSE": [np.sqrt(mean_squared_error(test_set[param], pred_test_df[param])) for param in parameters],
    "最佳超参数": [best_params[p] for p in parameters]
})

model_info = os.path.join(base_dir, "lightgbm_model.xlsx")
result_df.to_excel(model_info, index=False)
# 记录模型参数
lgbm_out = os.path.join(base_dir, "lgbm_amt.joblib")
joblib.dump(models["amt"], lgbm_out)

eval_df = pd.DataFrame(index=test_set.index)
eval_df["y_true"] = test_set[param]
eval_df["y_pred"] = pred_test_df[param]
eval_df["abs_err"] = (eval_df["y_pred"] - eval_df["y_true"]).abs()
eval_df["rel_err"] = eval_df["abs_err"] / np.maximum(np.abs(eval_df["y_true"]), 1e-6)

eval_df.to_excel(pred_y_path, index=True)

# pred_test_df.to_excel(pred_y_path, index=True)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001015 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4328
[LightGBM] [Info] Number of data points in the train set: 642, number of used features: 34
[LightGBM] [Info] Start training from score 136547862.317757
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4328
[LightGBM] [Info] Number of data points in the train set: 642, number of used features: 34
[LightGBM] [Info] Start training from score 136547862.317757
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [9]:
# # 加载
# loaded_model = joblib.load(lgbm_out)

# # 继续预测
# pred_test = loaded_model.predict(test_set[feature_cols])

In [10]:
# pd.DataFrame({
#     "feature": feature_cols,
#     "importance": models["amt"].feature_importances_
# }).sort_values("importance", ascending=False)